# CI failure prediction on GitHub Actions - exploratory data analysis

This notebook looks at the **pooled** per-run table: the enabled targets of `config/targets.json`, built in `collection.ipynb` and analysed as if they came from one CI workflow. It shows what the features look like, which of them separate failed from passed runs, how they correlate, how the failure rate moves over time, which jobs fail, and how easily the sources can be told apart. Next to every pooled view, the same statistic is given per source (target slug).

**Peeking rule.** Anything that relates features to the **label** is computed on the **dev segment** only (the first 20% of the pooled runs), the same rows later used for feature selection. Label-free summaries (missing values, value ranges, drift of feature values) use all rows. The few label-based views on later rows are marked *descriptive only*: they explain the data in the report and drive no choice.

**Main target: T2** (first failures); T1 is shown next to it where the contrast matters.

In [ ]:
import re

import numpy as np
import pandas as pd
import plotly.express as px
from IPython.display import Markdown, display
from sklearn.tree import DecisionTreeClassifier, export_text

from acise_p2_cidt.analysis.features import clean
from acise_p2_cidt.collect.targets import load_targets, slug
from acise_p2_cidt.data.load import load, pool_name, table_path
from acise_p2_cidt.training.cv import folds
from acise_p2_cidt.viz import plots

pd.set_option("display.max_rows", 100)

SOURCES = [slug(c.target) for c in load_targets()]
DATASET = pool_name(SOURCES)

t1, t2 = load(SOURCES, "T1"), load(SOURCES, "T2")
sizes = pd.concat({ds.meta["target"]: pd.DataFrame(ds.meta["per_source"]).T for ds in (t1, t2)}, axis=1)
sizes.loc["all"] = sizes.sum()
display(
    Markdown(f"**Pooled dataset `{DATASET}`**: rows and failures per source and target. The dev segment is the "
             f"first 20% of the pooled rows ({t1.meta['n_dev']} T1 rows, {t2.meta['n_dev']} T2 rows)."),
    sizes,
)

In [ ]:
def frame(ds) -> pd.DataFrame:
    """
    Features + label (pass / fail) + source + period, one row per run. Period: `dev`,
    `fold k` (the CV test segments), or `train only` (after dev, before fold 1).
    """
    period = np.where(ds.dev_mask, "dev", "train only").astype(object)
    for f in folds(ds):
        period[f.test] = f"fold {f.k + 1}"
    return ds.X.assign(
        failed=ds.y, label=ds.y.map({0: "pass", 1: "fail"}), source=ds.source, period=period, created_at=ds.order
    )

def profile(X: pd.DataFrame) -> pd.DataFrame:
    out = X.describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).T
    out["nan_%"] = X.isna().mean() * 100
    out["zero_%"] = (X == 0).mean() * 100
    out["nunique"] = X.nunique()
    out["skew"] = X.skew()
    return out.round(4)

f1, f2 = frame(t1), frame(t2)
PERIODS = ["dev", "train only", *[f"fold {k}" for k in range(1, 6)]]
display(
    Markdown("**T2 runs per source and period**: dev segment, rows only used for training, and the 5 CV test folds."),
    pd.crosstab(f2["source"], f2["period"]).reindex(columns=PERIODS, fill_value=0),
)

## 1. Feature profile (all rows, label-free)

One row per feature: range, percentiles, missing share, share of zeros, number of distinct values and skew, on the pooled T2 rows. Missing values come from history windows that are empty (e.g. `fail_rate_1h` when no run completed in the last hour), from the first runs of a lineage, an author or a file, and from run heads that could not be fetched (no git features). The second table gives the missing share per source: the sources differ in history depth and in how many heads are unreachable. Trees need no scaling, so heavy skew only matters for the plots (log axes).

In [ ]:
prof = profile(t2.X)
nan_by_source = (t2.X.isna().groupby(t2.source).mean().T * 100).round(1)
display(
    Markdown("**Feature profile, pooled T2, all rows**: range, percentiles, missing and zero share, distinct values "
             "and skew, sorted by missing share."),
    prof.sort_values("nan_%", ascending=False),
    Markdown("**Missing values per source** (T2, % of rows), features with any missing value."),
    nan_by_source[nan_by_source.max(axis=1) > 0],
)

### Features that carry no information

`clean()` (the first step of feature selection) drops mostly-missing (> 30% NaN), near-constant (variance of percentile ranks), and rare two-valued (< 1% minority) columns on the pooled dev rows. Listing them here shows which planned features are empty on the pool.

In [ ]:
kept = clean(t2.X[t2.dev_mask])
dropped = sorted(set(t2.X.columns) - set(kept.columns))
display(
    Markdown(f"**Features dropped by `clean()`** on the pooled T2 dev rows ({len(dropped)} of {t2.X.shape[1]})"),
    prof.loc[dropped, ["nan_%", "zero_%", "nunique", "mean"]],
)

## 2. Failure rate over time - T1 vs. T2

Weekly runs and failure rate of the pooled T2 runs (compare with the T1 plot in `collection.ipynb`); the dashed line ends the dev segment. The second plot gives the weekly T2 failure rate per source. The table gives runs and failure rate per period, per source and pooled (`all`), for T1 and T2; rates after the dev segment are descriptive only.

In [ ]:
dev_end = t2.order.iloc[t2.meta["n_dev"]]

fig = plots.runs_per_week(f2)
fig.add_vline(x=dev_end, line_dash="dash")
fig.update_layout(
    title="Runs and failure rate per week, pooled T2 (dashed: end of the dev segment)",
    yaxis2_tickformat=".0%", legend={"orientation": "h", "y": 1.1},
)
display(plots.save(fig, "eda_runs_per_week_T2"))

weekly = (
    f2.groupby(["source", pd.Grouper(key="created_at", freq="W")])["failed"]
    .agg(runs="size", fail_rate="mean")
    .reset_index()
)
fig = px.line(
    weekly, x="created_at", y="fail_rate", color="source", hover_data=["runs"],
    title="Weekly failure rate per source (T2)", labels={"created_at": "week", "fail_rate": "failure rate"},
)
display(plots.save(fig.update_layout(yaxis_tickformat=".0%"), "eda_fail_rate_per_source_T2"))

In [ ]:
period_table = pd.concat(
    {
        name: pd.concat([f, f.assign(source="all")])
        .groupby(["source", "period"], sort=False)["failed"]
        .agg(runs="size", fail_rate="mean")
        for name, f in [("T1", f1), ("T2", f2)]
    },
    axis=1,
).round(3)
display(
    Markdown("**Runs and failure rate per period**, per source and pooled (`all`), T1 and T2 (the folds of T1 and T2 "
             "cover different dates)."),
    period_table,
)

## 3. Failed-job categories (EDA only)

Which part of the CI fails. The job names of every source are mapped to four generic categories (`lint`, `test`, `build`, `other`) by one rule set, tried in that order. Job names are specific to each repository, so the category is **never** a feature. A failed run can fail several jobs, so shares are per failed job (all runs, T1).

In [ ]:
CATEGORIES = [
    ("lint", r"lint|clippy|fmt|format|tidy|typecheck|type check|spell|audit"),
    ("test", r"test|spec|e2e|wpt"),
    ("build", r"build|check|compile|install|deps|verify"),
]


def job_category(name: str) -> str:
    return next((cat for cat, pat in CATEGORIES if re.search(pat, name, re.I)), "other")


jobs = pd.concat(
    [pd.read_csv(table_path(s), usecols=["failed", "failed_job_names"]).assign(source=s) for s in SOURCES],
    ignore_index=True,
)
jobs = (
    jobs.loc[jobs["failed"] == 1]
    .assign(job=lambda d: d["failed_job_names"].fillna("").str.split("|"))
    .explode("job")
    .query("job != ''")
)
jobs["category"] = jobs["job"].map(job_category)
job_share = pd.crosstab(jobs["category"], jobs["source"], normalize="columns")
display(
    Markdown("**Most frequent failed job names per category** (all sources, top 5)"),
    jobs.groupby("category")["job"].agg(lambda js: ", ".join(js.value_counts().index[:5])).to_frame("job names"),
    Markdown("**Share of failed jobs per category and source** (each column sums to 1)"),
    job_share.round(3),
)
plots.save(
    px.bar(job_share.T, barmode="group", range_y=(0, 1)).update_layout(
        title="Share of failed jobs per category and source", yaxis_title="share of failed jobs"
    ),
    "eda_failed_job_share",
)

## 4. Which features separate failures? (dev segment)

Spearman correlation of every non-binary feature kept by `clean()` with the label, on the pooled dev rows, for T2 and T1. Rank correlation fits trees: a tree only uses the order of a feature's values. Binary features are compared by failure rate instead (section 4.2).

The heatmap repeats the correlation of the 10 strongest pooled features per source, on that source's dev rows: a feature whose sign or strength changes between sources does not mean the same thing in every repository.

On T2 the previous run of the lineage passed by definition, so it is also the last green run: every `*_since_green` feature equals its `*_since_prev_run` twin.

In [ ]:
def label_rank(f: pd.DataFrame) -> pd.Series:
    cols = [c for c in kept.columns.union(["prev_failed"]) if c in f and f[c].nunique() > 2]
    return f[cols].corrwith(f["failed"], method="spearman")

dev2 = f2[t2.dev_mask]

ranks = pd.DataFrame({"T2": label_rank(dev2), "T1": label_rank(f1[t1.dev_mask])})
ranks = ranks.reindex(ranks["T2"].abs().sort_values(ascending=False).index)
TOP = ranks.index[:10].tolist()
by_source = pd.DataFrame({s: g[TOP].corrwith(g["failed"], method="spearman") for s, g in dev2.groupby("source")})
display(
    Markdown("**Spearman correlation with the label**, pooled dev rows, T2 and T1, sorted by |T2| (top 25)"),
    ranks.round(4).head(25),
)
plots.save(
    px.imshow(by_source.round(2), zmin=-0.5, zmax=0.5, color_continuous_scale=plots.COLOR_SCALE, text_auto=True)
    .update_layout(height=600, title="Spearman with the label per source, top 10 pooled features (T2, dev rows)"),
    "eda_label_corr_per_source_T2",
)

In [ ]:
fig = plots.dist_by_label(dev2, TOP, label="label", kind="box", log_y=True)  # zeros vanish on a log axis
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.update_yaxes(dtick=1)  # one labelled tick per decade
plots.save(fig.update_layout(height=1000, title="Top features by label (pooled T2, dev segment, log axis)"), "eda_top_features_T2")

### 4.2 Binary features

Share of runs where each flag is 1, per period (label-free), and the dev failure rate when it is 1 vs. 0, pooled and per source. The flags include the event (`event_*`), `from_fork` and `lineage_first_run`. A flag that is almost never 1 on the dev rows cannot be learned from them.

In [ ]:
binary = [c for c in t2.X.columns if t2.X[c].nunique() == 2 and t2.X[c].dropna().isin([0, 1]).all()]


def rate_if(g: pd.DataFrame) -> pd.DataFrame:
    """Failure rate of the rows where each flag is 1, and where it is 0."""
    return pd.DataFrame(
        {
            "if 1": [g.loc[g[c] == 1, "failed"].mean() for c in binary],
            "if 0": [g.loc[g[c] == 0, "failed"].mean() for c in binary],
        },
        index=binary,
    )


share = f2.groupby("period", sort=False)[binary].mean().T.add_prefix("share_1: ")
share["dev runs with 1"] = dev2[binary].sum()
rates = pd.concat({"all": rate_if(dev2), **{s: rate_if(g) for s, g in dev2.groupby("source")}}, axis=1)
display(
    Markdown("**Binary features, pooled T2**: share of runs with value 1 per period (label-free), and dev runs with 1"),
    share.round(4),
    Markdown("**Dev failure rate when the flag is 1 vs. 0**, pooled (`all`) and per source (T2)"),
    rates.round(4),
)

## 5. Correlations between features (dev segment, T2)

Window families (`fail_rate_*`, `ci_load_*`, `*_since_*`) are built to overlap. The heatmap and the list of near-duplicate pairs show how much; feature selection prunes pairs above rho = 0.8.

In [ ]:
corr = kept.corr(method="spearman")
fig = plots.corr_heatmap(corr)
fig.update_xaxes(dtick=1, tickfont_size=8).update_yaxes(dtick=1, tickfont_size=8)
display(plots.save(fig.update_layout(height=1100, width=1100, title="Spearman correlation (pooled T2, dev segment)"), "eda_corr_T2"))

c = corr.abs()
pairs = c.where(np.triu(np.ones(c.shape, bool), 1)).stack()
display(
    Markdown(f"**Feature pairs with |rho| > 0.8** (pooled T2, dev segment): {(pairs > 0.8).sum()} of {len(pairs)} pairs"),
    pairs[pairs > 0.8].sort_values(ascending=False).to_frame("abs_spearman").round(4),
)

## 6. Drift across periods

**Feature values (label-free).** Median of each feature per period (dev, then the CV test folds), for the features whose median moves most relative to the dev segment. A tree trained on early rows sees thresholds in early units; growing CI load or repository size shifts them. The second table gives the median of the top label features per source: sources differ in scale (durations, load, churn), which a tree can use to tell them apart.

In [ ]:
med = f2.groupby("period", sort=False)[kept.columns.tolist()].median().T
shift = (med.drop(columns="dev").sub(med["dev"], axis=0)).abs().max(axis=1) / med["dev"].abs().replace(0, np.nan)
display(
    Markdown("**Median per period, pooled T2**: the 15 features whose median moves most relative to the dev segment"),
    med.loc[shift.sort_values(ascending=False).index[:15]].round(3),
    Markdown("**Median per source, T2**, the 10 strongest label features: the scale of each source"),
    f2.groupby("source")[TOP].median().T.round(3),
)

**Feature-label relation per period (descriptive only).** The Spearman correlation of the top dev features with the label, per period. It shows whether the signal found on the dev rows holds later. Nothing here feeds feature selection or tuning.

In [ ]:
per_period = pd.DataFrame(
    {p: g[TOP].corrwith(g["failed"], method="spearman") for p, g in f2.groupby("period", sort=False)}
)
fig = px.imshow(per_period.round(2), zmin=-0.5, zmax=0.5, color_continuous_scale=plots.COLOR_SCALE, text_auto=True)
plots.save(fig.update_layout(height=600, title="Spearman with the label per period (pooled T2, descriptive)"), "eda_label_corr_per_period_T2")

## 7. Repository proxy

A depth-3 tree that predicts the **source** of a run from its features, on the pooled T2 dev rows: fitted on the first 70% and scored on the last 30%. High accuracy means the sources are easy to tell apart, so the failure model could learn the source first and then predict its base rate. The accuracy is compared with always predicting the most frequent source; the splits show which features give a source away.

In [ ]:
Xd = t2.X.loc[t2.dev_mask, kept.columns]
sd = t2.source[t2.dev_mask]
cut = int(len(Xd) * 0.7)
proxy = DecisionTreeClassifier(max_depth=3, random_state=0).fit(Xd.iloc[:cut], sd.iloc[:cut])
acc = float((proxy.predict(Xd.iloc[cut:]) == sd.iloc[cut:]).mean())
base = float(sd.iloc[cut:].value_counts(normalize=True).iloc[0])
display(Markdown(f"**Repository proxy** (T2 dev rows, last 30%): accuracy {acc:.3f}, most frequent source {base:.3f}"))
print(export_text(proxy, feature_names=list(Xd.columns)))

## Observations

_To be written after the run on the pooled data._